Proceso de lectura de archivos CSV

In [1]:
import pandas as pd
import requests

In [2]:
df=pd.read_csv("Titanic.csv")
print(f'total lineas importadas:  {len(df)}')

total lineas importadas:  1309


In [5]:
def leer_datos_csv():
        source="Titanic.csv"
        df=pd.read_csv(source)
        print(f'total lineas importadas:  {len(df)}')
        return df

In [7]:
df=leer_datos_csv()

total lineas importadas:  1309


In [2]:
subject='cooking'
url = f"https://openlibrary.org/subjects/{subject}.json?limit=50"
response = requests.get(url)
data = response.json()
df = pd.json_normalize(data['works'])
df = df[['title', 'key', 'first_publish_year']]
print(f"Batch pulled {len(df)} book records.")

Batch pulled 50 book records.


In [3]:
df.head(10)

,title,key,first_publish_year
0,Kokuritsu Kokkai Toshokan shozō Meijiki kankō ...,/works/OL33207850W,1991
1,Frugal housewife,/works/OL107969W,1829
2,Boston Cooking-School cook book,/works/OL1798750W,1896
3,Como agua para chocolate,/works/OL953162W,1989
4,The Virginia house-wife,/works/OL2653703W,1824
5,On Cooking,/works/OL1874827W,1994
6,Old Cookery Books and Ancient Cuisine,/works/OL1086812W,1886
7,Joy of Cooking,/works/OL267968W,1931
8,The Bartender's Guide,/works/OL7418388W,1862
9,国立 国会 図書館 所蔵 明治期 刋行 図書 マイクロ版 集成,/works/OL39080200W,1991


In [4]:
from pipeline import run_orchestator

In [5]:
ad1 = run_orchestator()

--- Lectura de csv
total lineas importadas:  1309
--- Lectura de titulos libros
Batch pulled 10 book records.
--- Lectura del clima en tiempo real
  > instantanea 1...
  > instantanea 2...
  > instantanea 3...
  > instantanea 4...
  > instantanea 5...
--- Resumen de datos sin transformar

FUENTE: Titanic
Rows: 1309 | Columns: ['Passengerid', 'Age', 'Fare', 'Sex', 'sibsp', 'zero', 'zero.1', 'zero.2', 'zero.3', 'zero.4', 'zero.5', 'zero.6', 'Parch', 'zero.7', 'zero.8', 'zero.9', 'zero.10', 'zero.11', 'zero.12', 'zero.13', 'zero.14', 'Pclass', 'zero.15', 'zero.16', 'Embarked', 'zero.17', 'zero.18', '2urvived']
   Passengerid   Age     Fare  Sex  sibsp  zero  zero.1  zero.2  zero.3  \
0            1  22.0   7.2500    0      1     0       0       0       0   
1            2  38.0  71.2833    1      1     0       0       0       0   

   zero.4  ...  zero.12  zero.13  zero.14  Pclass  zero.15  zero.16  Embarked  \
0       0  ...        0        0        0       3        0        0       2.0 

In [16]:
df=ad1['Titanic']

In [9]:
print(df.groupby("2urvived").size())

2urvived
0    967
1    342
dtype: int64


In [11]:
df_batch=ad1['Libros']
df_batch['UniqueKey']=df_batch["key"].str.split('/').str[-1]

In [12]:
ad1['Libros']=df_batch

Titanic: limpieza de datos

In [12]:
# reemplazo de datos: por la media
import pandas as pd
import numpy as np

# Calculo de la media de valores que no son 0
media_fare = df.loc[df['Fare'] != 0, 'Fare'].mean()

# Reemplazar los valores 0 con la media calculada antes
df['Fare'] = df['Fare'].replace(0, media_fare)

In [ ]:
# Imputación de datos con KNN
from sklearn.impute import KNNImputer

In [17]:
from sklearn.impute import KNNImputer


# 1. Convertir 0 a NaN para que el imputador los identifique como faltantes
df['Fare'] = df['Fare'].replace(0, np.nan)

# 2. Crear instancia de objeto Imputer
imputer = KNNImputer(n_neighbors=5)

# 3. Aplicar el imputador
# Se utilizan los otros valores números como guia de la imputación
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

# 4. Reemplazar la columna original con la nueva columna de valores imputados
#df['Fare'] = df_imputed['Fare']